# Дообучение rubert-tiny2 на разметке «Повестей Белкина»

Блокнот запускает `archword.finetune`: модель учится по паре «слово, предложение» оценивать, насколько слово трудное.
Перед запуском: **Среда выполнения → Сменить тип среды → GPU (T4)**. Дальше выполняй ячейки по порядку.
Тест (раунд 3) используется один раз, настройки не подбираются под него.

In [ ]:
!nvidia-smi -L || echo "GPU не найден: обучение пойдёт на CPU и займёт намного дольше" 

## 1. Код и зависимости

In [ ]:
!rm -rf archaic-words
!git clone https://github.com/borisovv1905-art/archaic-words.git
%cd archaic-words
!pip install -q -e . transformers

## 2. Публичные данные (Библия, Zenodo 7419194)
Ячейка пробует скачать архив сама. Если не получилось, скачай `lexical_complexity.zip` на странице https://zenodo.org/records/7419194 и загрузи его через значок папки слева, в корень `/content/archaic-words`.

In [ ]:
import os, zipfile, subprocess, glob
os.makedirs("data/raw", exist_ok=True)
if not glob.glob("data/raw/lexical_complexity/word_complexity_ru_train.tsv"):
    if not os.path.exists("lexical_complexity.zip"):
        subprocess.run(["wget", "-q", "-O", "lexical_complexity.zip",
                        "https://zenodo.org/records/7419194/files/lexical_complexity.zip?download=1"])
    assert os.path.exists("lexical_complexity.zip") and os.path.getsize("lexical_complexity.zip") > 10000, \
        "Архив не скачался: загрузи lexical_complexity.zip вручную (см. текст выше)"
    zipfile.ZipFile("lexical_complexity.zip").extractall("data/raw")
print(glob.glob("data/raw/**/*.tsv", recursive=True))

Если файлы оказались в папке на уровень глубже (например `data/raw/lexical_complexity/lexical_complexity/...`), переложи их так, чтобы пути были
`data/raw/lexical_complexity/word_complexity_ru_train.tsv` и `..._test.tsv`.

In [ ]:
# быстрая проверка, что всё на месте
!ls data/raw/lexical_complexity data/annotation
!python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())" 

## 3. Основной запуск
Четыре постановки (только Библия, только своя разметка, сначала Библия потом своя, всё вместе) × 3 сида. Должно занять порядка 10-20 минут на T4.
Проверочный столбец `public_test_spearman` - контроль: модель, обученная на Библии, должна давать на публичном тесте Spearman около 0,5-0,6 (частота + длина дают 0,55). Если там почти 0, значит что-то сломано.

In [ ]:
!python -m archword.finetune --raw data/raw --out results/finetune_round3.json

## 4. Результаты

In [ ]:
import json
r = json.load(open("results/finetune_round3.json"))
print("слов в тесте:", r["n_test"], "| своей разметки в обучении:", r["n_own_train"])
for name, s in r["settings"].items():
    print(f"{name:16s} spearman {s['spearman']:.3f} {s['spearman_ci95']} | AUC>=2 {s.get('auc_label_ge2')} | AUC>=3 {s.get('auc_label_ge3')} | по сидам {s['spearman_per_seed']} | публичный тест {s.get('public_test_spearman')}")

Для сравнения (те же 196 слов раунда 3, `results/transfer_rounds_lemma.json`): частота формы Spearman 0,17, AUC>=2 0,62; частота леммы 0,20 и 0,64.

## 5. Скачать результаты
Файлы `results/finetune_round3.json` и `results/finetune_round3.predictions.csv` нужно прислать мне или положить в папку проекта.

In [ ]:
from google.colab import files
files.download("results/finetune_round3.json")
files.download("results/finetune_round3.predictions.csv")